# Gaze Data Visualization
Quick visualization of eye gaze tracking data from the VR demo

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from scipy.stats import gaussian_kde
import os

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 2. Load Gaze Data

In [ ]:
import glob, os

# Auto-find newest gaze CSV in the same folder
here = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
csv_files = sorted(glob.glob(os.path.join(here, "gaze_*.csv")), key=os.path.getmtime, reverse=True)
data_file = csv_files[0] if csv_files else "gaze_20260416_155724.csv"
print(f"Loading: {os.path.basename(data_file)}")

df = pd.read_csv(data_file)
print(f"Loaded {len(df)} gaze samples  |  duration: {df.time_s.max():.1f} s  |  {len(df.columns)} columns")
print("Columns:", list(df.columns))


## 3. Explore Data Structure

In [ ]:
print("Data shape:", df.shape)
print("\nColumn names and types:")
print(df.dtypes)
print("\nFirst few rows:")
print(df.head())
print("\nData summary:")
print(df.describe())
print("\nMissing values:")
print(df.isnull().sum())

## 4. Visualize Gaze Direction Points

In [ ]:
fig = plt.figure(figsize=(14, 5))

# 3D gaze direction vectors
ax1 = fig.add_subplot(121, projection='3d')
scatter = ax1.scatter(df['gaze_x'], df['gaze_y'], df['gaze_z'], 
                     c=df['time_s'], cmap='viridis', s=20, alpha=0.6)
ax1.set_xlabel('Gaze X')
ax1.set_ylabel('Gaze Y')
ax1.set_zlabel('Gaze Z')
ax1.set_title('3D Gaze Direction Vectors (colored by time)')
cbar = plt.colorbar(scatter, ax=ax1, pad=0.1)
cbar.set_label('Time (s)')

# 2D gaze direction projection (X-Y plane)
ax2 = fig.add_subplot(122)
ax2.scatter(df['gaze_x'], df['gaze_y'], c=df['time_s'], 
           cmap='viridis', s=20, alpha=0.6)
ax2.set_xlabel('Gaze X')
ax2.set_ylabel('Gaze Y')
ax2.set_title('2D Gaze Direction (X-Y plane)')
ax2.grid(True, alpha=0.3)
plt.colorbar(ax2.scatter(df['gaze_x'], df['gaze_y'], c=df['time_s'], 
            cmap='viridis', s=20, alpha=0.6), ax=ax2, label='Time (s)')

plt.tight_layout()
plt.show()

## 5. Create Heatmap Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Heatmap: Gaze X-Y distribution (using time as color)
ax = axes[0, 0]
scatter = ax.scatter(df['gaze_x'], df['gaze_y'], c=df['time_s'], s=30, cmap='hot', alpha=0.6)
ax.set_xlabel('Gaze X')
ax.set_ylabel('Gaze Y')
ax.set_title('Gaze Distribution (X-Y, colored by time)')
plt.colorbar(scatter, ax=ax, label='Time (s)')

# Heatmap: Gaze X-Z distribution (using time as color)
ax = axes[0, 1]
scatter = ax.scatter(df['gaze_x'], df['gaze_z'], c=df['time_s'], s=30, cmap='hot', alpha=0.6)
ax.set_xlabel('Gaze X')
ax.set_ylabel('Gaze Z')
ax.set_title('Gaze Distribution (X-Z, colored by time)')
plt.colorbar(scatter, ax=ax, label='Time (s)')

# Head position in 2D
ax = axes[1, 0]
ax.scatter(df['head_x'], df['head_z'], c=df['time_s'], s=20, 
          cmap='plasma', alpha=0.7)
ax.set_xlabel('Head X')
ax.set_ylabel('Head Z')
ax.set_title('Head Position Trajectory (X-Z plane)')
ax.grid(True, alpha=0.3)

# Head Y position over time
ax = axes[1, 1]
ax.plot(df['time_s'], df['head_y'], linewidth=2, color='steelblue')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Head Y (height)')
ax.set_title('Head Height Over Time')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Plot Gaze Trajectory Over Time

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Gaze X component over time
ax = axes[0]
ax.plot(df['time_s'], df['gaze_x'], linewidth=1.5, color='red', alpha=0.7)
ax.fill_between(df['time_s'], df['gaze_x'], alpha=0.3, color='red')
ax.set_ylabel('Gaze X (left-right)')
ax.set_title('Gaze Direction Components Over Time')
ax.grid(True, alpha=0.3)

# Gaze Y component over time
ax = axes[1]
ax.plot(df['time_s'], df['gaze_y'], linewidth=1.5, color='green', alpha=0.7)
ax.fill_between(df['time_s'], df['gaze_y'], alpha=0.3, color='green')
ax.set_ylabel('Gaze Y (up-down)')
ax.grid(True, alpha=0.3)

# Gaze Z component over time
ax = axes[2]
ax.plot(df['time_s'], df['gaze_z'], linewidth=1.5, color='blue', alpha=0.7)
ax.fill_between(df['time_s'], df['gaze_z'], alpha=0.3, color='blue')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Gaze Z (forward-back)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Generate Summary Statistics

In [ ]:
# Calculate gaze direction magnitude
df['gaze_magnitude'] = np.sqrt(df['gaze_x']**2 + df['gaze_y']**2 + df['gaze_z']**2)

# Calculate head position magnitude
df['head_distance'] = np.sqrt(df['head_x']**2 + df['head_y']**2 + df['head_z']**2)

# Calculate gaze velocity (rate of change)
df['gaze_velocity'] = np.sqrt(
    np.diff(df['gaze_x'], prepend=0)**2 + 
    np.diff(df['gaze_y'], prepend=0)**2 + 
    np.diff(df['gaze_z'], prepend=0)**2
) / np.diff(df['time_s'], prepend=df['time_s'].iloc[0])

print("="*60)
print("GAZE DATA SUMMARY STATISTICS")
print("="*60)

print(f"\nRecording Duration: {df['time_s'].max():.2f} seconds")
print(f"Number of Samples: {len(df)}")
print(f"Sample Rate: {len(df) / df['time_s'].max():.1f} Hz")

print("\n--- GAZE DIRECTION STATISTICS ---")
print(f"Gaze X:  mean={df['gaze_x'].mean():.3f}, std={df['gaze_x'].std():.3f}")
print(f"Gaze Y:  mean={df['gaze_y'].mean():.3f}, std={df['gaze_y'].std():.3f}")
print(f"Gaze Z:  mean={df['gaze_z'].mean():.3f}, std={df['gaze_z'].std():.3f}")
print(f"Gaze Magnitude: mean={df['gaze_magnitude'].mean():.3f}, max={df['gaze_magnitude'].max():.3f}")

print("\n--- HEAD POSITION STATISTICS ---")
print(f"Head X:  mean={df['head_x'].mean():.3f}, range=[{df['head_x'].min():.3f}, {df['head_x'].max():.3f}]")
print(f"Head Y:  mean={df['head_y'].mean():.3f}, range=[{df['head_y'].min():.3f}, {df['head_y'].max():.3f}]")
print(f"Head Z:  mean={df['head_z'].mean():.3f}, range=[{df['head_z'].min():.3f}, {df['head_z'].max():.3f}]")
print(f"Head Distance: mean={df['head_distance'].mean():.3f}, std={df['head_distance'].std():.3f}")

print("\n--- GAZE MOTION STATISTICS ---")
gaze_vel = df['gaze_velocity'].replace([np.inf, -np.inf], np.nan)
print(f"Gaze Velocity: mean={gaze_vel.mean():.4f}, max={gaze_vel.max():.4f}")

print("\n" + "="*60)

## 8. Head Movement Analysis


In [ ]:
## Head Movement Trajectory
# Gaze is constant (proxy mode = HMD forward). Head position has real movement.

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f"Head Movement  —  {os.path.basename(data_file)}", fontsize=13, fontweight="bold")

# --- XZ trajectory coloured by time ---
ax = axes[0]
sc = ax.scatter(df["head_x"], df["head_z"], c=df["time_s"], cmap="plasma", s=6, alpha=0.8)
ax.scatter(df["head_x"].iloc[0],  df["head_z"].iloc[0],  marker="o", s=80, color="lime",  zorder=5, label="Start")
ax.scatter(df["head_x"].iloc[-1], df["head_z"].iloc[-1], marker="X", s=80, color="red",   zorder=5, label="End")
plt.colorbar(sc, ax=ax, label="Time (s)")
ax.set_xlabel("Head X (m)"); ax.set_ylabel("Head Z (m)")
ax.set_title("XZ Trajectory (top-down view)"); ax.set_aspect("equal"); ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- Head X and Z over time ---
ax = axes[1]
ax.plot(df["time_s"], df["head_x"], label="head_x (left/right)",  color="tab:red",  lw=1.0)
ax.plot(df["time_s"], df["head_z"], label="head_z (forward/back)", color="tab:blue", lw=1.0)
ax.set_xlabel("Time (s)"); ax.set_ylabel("Position (m)")
ax.set_title("Head X / Z over time"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# --- Speed over time ---
dt   = df["time_s"].diff().replace(0, np.nan).fillna(1/90)
dx   = df["head_x"].diff().fillna(0)
dz   = df["head_z"].diff().fillna(0)
speed = (np.sqrt(dx**2 + dz**2) / dt).rolling(9, center=True, min_periods=1).mean()
ax = axes[2]
ax.plot(df["time_s"], speed, color="tab:green", lw=1.0)
ax.fill_between(df["time_s"], speed, alpha=0.25, color="tab:green")
ax.set_xlabel("Time (s)"); ax.set_ylabel("Speed (m/s)")
ax.set_title("Head movement speed"); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Total distance: {np.sqrt(dx**2 + dz**2).sum():.2f} m")
print(f"Mean speed:     {speed.mean():.3f} m/s")
print(f"Max speed:      {speed.max():.3f} m/s")


## 9. Pupil Size Analysis (if available)


In [ ]:
# Check if pupil size data is available
if 'pupil_size_mm' in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Pupil size over time
    ax = axes[0, 0]
    ax.plot(df['time_s'], df['pupil_size_mm'], linewidth=2, color='purple', alpha=0.7)
    ax.fill_between(df['time_s'], df['pupil_size_mm'], alpha=0.3, color='purple')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Pupil Size (mm)')
    ax.set_title('Pupil Size Over Time')
    ax.grid(True, alpha=0.3)
    
    # Pupil size distribution
    ax = axes[0, 1]
    ax.hist(df['pupil_size_mm'], bins=30, color='purple', alpha=0.7, edgecolor='black')
    ax.set_xlabel('Pupil Size (mm)')
    ax.set_ylabel('Frequency')
    ax.set_title('Pupil Size Distribution')
    ax.axvline(df['pupil_size_mm'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["pupil_size_mm"].mean():.2f}mm')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    # Pupil size vs gaze magnitude
    if 'gaze_magnitude' in df.columns:
        ax = axes[1, 0]
        scatter = ax.scatter(df['gaze_magnitude'], df['pupil_size_mm'], c=df['time_s'], cmap='viridis', s=20, alpha=0.6)
        ax.set_xlabel('Gaze Magnitude')
        ax.set_ylabel('Pupil Size (mm)')
        ax.set_title('Pupil Size vs Gaze Magnitude')
        plt.colorbar(scatter, ax=ax, label='Time (s)')
        ax.grid(True, alpha=0.3)
    
    # Pupil size and head height correlation
    ax = axes[1, 1]
    ax2 = ax.twinx()
    ax.plot(df['time_s'], df['pupil_size_mm'], color='purple', linewidth=2, label='Pupil Size', alpha=0.7)
    ax2.plot(df['time_s'], df['head_y'], color='orange', linewidth=2, label='Head Height', alpha=0.7)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Pupil Size (mm)', color='purple')
    ax2.set_ylabel('Head Height (m)', color='orange')
    ax.set_title('Pupil Size & Head Height Over Time')
    ax.tick_params(axis='y', labelcolor='purple')
    ax2.tick_params(axis='y', labelcolor='orange')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print pupil size statistics
    print("\n--- PUPIL SIZE STATISTICS ---")
    print(f"Pupil Size: mean={df['pupil_size_mm'].mean():.2f}mm, std={df['pupil_size_mm'].std():.2f}mm")
    print(f"Pupil Size: min={df['pupil_size_mm'].min():.2f}mm, max={df['pupil_size_mm'].max():.2f}mm")
else:
    print("Pupil size data not available in this recording.")
    print("To capture pupil size, upgrade to the Pimax PGEE SDK or OpenXR extension.")
    print("See GUIDE.md §8 for integration instructions.")